# build_startups_master

**Objective:** Merge the cleaned Crunchbase company and funding-round exports into a single firm-level master table, deriving funding aggregates, the most-advanced funding stage, and a tech-field grouping.

**Inputs:** `startup_sample_crunchbase_export.csv`; `funding_rounds_crunchbase_export.csv`.

**Outputs:** `startups_master.csv` (one row per company).

## Imports

In [1]:
# Imports
import numpy as np
import pandas as pd
from pathlib import Path

## Paths and observation window

In [2]:
# Paths and observation window
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
COMP_PATH  = RAW / 'crunchbase' / 'startup_sample_crunchbase_export.csv'
ROUND_PATH = RAW / 'crunchbase' / 'funding_rounds_crunchbase_export.csv'
OUT_PATH   = PROC / 'startups_master.csv'

OBS_END = pd.Timestamp('2026-04-30')

## Funding-stage USD aggregates

In [3]:
# Funding-stage buckets for early, mid, and late USD aggregates
EARLY = {'Pre-Seed', 'Seed', 'Angel', 'Series A'}
MID   = {'Series B', 'Venture - Series Unknown'}
LATE  = {'Series C', 'Series D', 'Series E', 'Series F', 'Series G', 'Series H', 'Series I',
         'Private Equity', 'Post-IPO Equity', 'Post-IPO Debt', 'Post-IPO Secondary'}
GRANT = {'Grant', 'Non-equity Assistance'}

EQUITY = {'Pre-Seed', 'Seed', 'Angel',
          'Series A', 'Series B', 'Series C', 'Series D', 'Series E',
          'Series F', 'Series G', 'Series H', 'Series I',
          'Private Equity', 'Post-IPO Equity', 'Post-IPO Secondary',
          'Venture - Series Unknown'}

In [4]:
# Equity-stage ranks for the most-advanced-round dummies
STAGE_RANK = {
    'Pre-Seed':            1,
    'Seed':                2,
    'Angel':               3,
    'Series A':            4,
    'Series B':            5,
    'Series C':            6, 'Series D': 6, 'Series E': 6,
    'Series F':            6, 'Series G': 6, 'Series H': 6, 'Series I': 6,
    'Private Equity':      7,
    'Post-IPO Equity':     8, 'Post-IPO Debt': 8, 'Post-IPO Secondary': 8,
}
RANK_TO_LABEL = {
    1: 'pre_seed', 2: 'seed', 3: 'angel', 4: 'series_a', 5: 'series_b',
    6: 'series_c_plus', 7: 'private_equity', 8: 'post_ipo',
}
STAGE_COLS = [f'stage_{label}' for label in RANK_TO_LABEL.values()]
BUCKET_EARLY_RANKS = {1, 2, 3, 4, 5}
BUCKET_LATE_RANKS  = {6, 7, 8}

## Tech-field keyword classifier

In [5]:
# Tech-field keyword lists (life sciences, ICT, cleantech, other)
LIFE = ['Biotechnology','Pharmaceutical','Medical','Health Care','Health Diagnostics','Therapeutics',
        'Life Science','Drug Discovery','Genetics','Bioinformatics','Mental Health','mHealth',
        'Wellness','Nutrition','Diabetes','Oncology','Neuroscience','Veterinary','Dental',
        'Hospital','Medical Device','Clinical Trials','Cosmetics','Personal Health']
ICT  = ['Software','Information Technology','Internet','Artificial Intelligence','Cloud Computing',
        'Cloud Storage','Cloud Data Services','Mobile','Mobile Apps','SaaS','Cyber Security','Big Data',
        'Machine Learning','Data Analytics','Computer','Network Hardware','Database','Developer Tools',
        'Telecommunications','Web Hosting','IT Infrastructure','FinTech','Financial Technology',
        'EdTech','PropTech','Blockchain','Cryptocurrency','VR','AR','Gaming','E-Commerce',
        'Social Network','Web Development','App Development','Search Engine','Analytics']
CLEAN= ['Clean Energy','Renewable Energy','Solar','Wind Energy','Electric Vehicle','Sustainability',
        'Green Consumer Goods','Recycling','Battery','Hydrogen','CleanTech','Environmental Engineering',
        'Climate','Waste Management','Water Purification','Smart Grid','Energy Storage',
        'Energy Efficiency','Biofuel','Geothermal']

In [6]:
# Classify a company by keyword priority into a tech field group
def classify_field(text: str) -> str:
    if pd.isna(text):
        return 'other'
    t = str(text).lower()
    if any(k.lower() in t for k in LIFE):  return 'life sciences'
    if any(k.lower() in t for k in ICT):   return 'ICT'
    if any(k.lower() in t for k in CLEAN): return 'cleantech'
    return 'other'

## Per-company round aggregation

In [7]:
# Aggregate one company's funding rounds into firm-level fields
def agg_company(g: pd.DataFrame) -> pd.Series:
    g_sorted = g.sort_values('Announced Date')
    first = g_sorted.iloc[0]

    ranks = g['_stage_rank'].dropna()
    if len(ranks):
        max_rank = int(ranks.max())
    else:
        max_rank = 0

    eq_dates = g.loc[g['_is_equity'], 'Announced Date']
    first_equity_date = eq_dates.min() if len(eq_dates) else pd.NaT

    out = {
        'total_funding_usd':         g['Money Raised (in USD)'].sum(min_count=1),
        'n_rounds_total':            len(g),
        'first_funding_date':        first['Announced Date'],
        'first_funding_type':        first['Funding Type'],
        'first_funding_stage':       first['Funding Stage'],
        'first_funding_amount_usd':  first['Money Raised (in USD)'],
        'received_grant_binary':     int(g['is_grant'].any()),
        'total_grants_usd':          g.loc[g['is_grant'], 'Money Raised (in USD)'].sum(min_count=1),
        'has_early_round_binary':    int(g['is_early'].any()),
        'first_equity_round_date':   first_equity_date,
        'total_raised_early_usd':    g.loc[g['is_early'], 'Money Raised (in USD)'].sum(min_count=1),
        'total_raised_mid_usd':      g.loc[g['is_mid'],   'Money Raised (in USD)'].sum(min_count=1),
        'total_raised_late_usd':     g.loc[g['is_late'],  'Money Raised (in USD)'].sum(min_count=1),
        '_max_stage_rank':           max_rank,
    }
    return pd.Series(out)

## Main build routine

In [8]:
# Main pipeline: merge companies and rounds into the master table
def main():
    comp   = pd.read_csv(COMP_PATH)
    rounds = pd.read_csv(ROUND_PATH)
    rounds['Announced Date'] = pd.to_datetime(rounds['Announced Date'], errors='coerce', format='mixed')

    r = rounds.copy()
    r['is_early'] = r['Funding Type'].isin(EARLY)
    r['is_mid']   = r['Funding Type'].isin(MID)
    r['is_late']  = r['Funding Type'].isin(LATE)
    r['is_grant'] = r['Funding Type'].isin(GRANT)
    r['_stage_rank'] = r['Funding Type'].map(STAGE_RANK)
    r['_is_equity']  = r['Funding Type'].isin(EQUITY)

    agg = r.groupby('Organization Name', as_index=False).apply(agg_company, include_groups=False)

    m = comp.merge(agg, on='Organization Name', how='left')

    zero_fill = ['n_rounds_total', 'received_grant_binary', 'has_early_round_binary']
    for c in zero_fill:
        m[c] = m[c].fillna(0).astype(int)
    m['_max_stage_rank'] = m['_max_stage_rank'].fillna(0).astype(int)

    for rank, label in RANK_TO_LABEL.items():
        m[f'stage_{label}'] = (m['_max_stage_rank'] == rank).astype(int)
    m['bucket_early'] = m['_max_stage_rank'].isin(BUCKET_EARLY_RANKS).astype(int)
    m['bucket_late']  = m['_max_stage_rank'].isin(BUCKET_LATE_RANKS).astype(int)
    m = m.drop(columns=['_max_stage_rank'])

    m['Founded Date'] = pd.to_datetime(m['Founded Date'], errors='coerce', format='mixed')
    m['Closed Date']  = pd.to_datetime(m['Closed Date'],  errors='coerce', format='mixed')
    m['Exit Date']    = pd.to_datetime(m['Exit Date'],    errors='coerce', format='mixed')
    m['founding_year'] = m['Founded Date'].dt.year.astype('Int64')
    m['country'] = m['Headquarters Location'].astype(str).str.split(',').str[-1].str.strip()
    m.loc[m['Headquarters Location'].isna(), 'country'] = pd.NA

    combined = m['Primary Industry'].fillna('') + ' | ' + m['Industries'].fillna('')
    m['tech_field_group'] = combined.apply(classify_field)

    m['failed_binary'] = (m['Operating Status'] == 'Closed').astype(int)

    ipo_set = {'Public', 'Delisted'}
    acq_set = {'Was Acquired', 'Made Acquisitions, Was Acquired'}
    m['exit_binary'] = (
        m['IPO Status'].isin(ipo_set) | m['Acquisition Status'].isin(acq_set)
    ).astype(int)

    def exit_type(row):
        if row['IPO Status'] in ipo_set: return 'IPO'
        if row['Acquisition Status'] in acq_set: return 'M&A'
        return 'none'
    m['exit_type'] = m.apply(exit_type, axis=1)

    m['years_to_exit'] = ((m['Exit Date'] - m['Founded Date']).dt.days / 365.25).round(2)

    m['first_funding_date'] = pd.to_datetime(m['first_funding_date'], errors='coerce')
    m['age_at_first_funding'] = ((m['first_funding_date'] - m['Founded Date']).dt.days / 365.25).round(2)

    m['observation_end_date'] = OBS_END
    m['age_at_observation_end'] = ((OBS_END - m['Founded Date']).dt.days / 365.25).round(2)

    m['log_total_funding_usd'] = np.log1p(m['total_funding_usd'])

    for c in ['Closed Date', 'Exit Date', 'first_funding_date',
              'first_equity_round_date', 'observation_end_date']:
        m[c] = pd.to_datetime(m[c], errors='coerce').dt.strftime('%Y-%m-%d')

    cols = [
        'Organization Name', 'Organization Name URL',
        'founding_year',
        'Headquarters Location', 'country',
        'Primary Industry', 'tech_field_group',
        'Operating Status', 'Closed Date', 'failed_binary',
        'Acquisition Status', 'Acquisition Type', 'IPO Status', 'Exit Date',
        'exit_binary', 'exit_type', 'years_to_exit',
        'total_funding_usd', 'log_total_funding_usd', 'n_rounds_total',
        'first_funding_date', 'first_funding_type', 'first_funding_stage',
        'first_funding_amount_usd', 'age_at_first_funding',
        'first_equity_round_date',
        *STAGE_COLS,
        'has_early_round_binary',
        'bucket_early', 'bucket_late',
        'total_raised_early_usd', 'total_raised_mid_usd', 'total_raised_late_usd',
        'received_grant_binary', 'total_grants_usd',
        'observation_end_date', 'age_at_observation_end',
    ]
    m = m[cols]
    m.to_csv(OUT_PATH, index=False)
    print(f'Wrote: {OUT_PATH}  ({m.shape[0]:,} rows x {m.shape[1]} cols)')

## Entry point

In [ ]:
# Entry point
if __name__ == '__main__':
    main()